# Out-of-Sample Inference (2024-2025)

This notebook demonstrates Phase 2:
1.  **Load Prompts**: Load the optimized agent instructions from Phase 1.
2.  **Inference**: Run the **World Model Inference** (Event-Driven) on future data (2024-2025). This iterates day-by-day, revealing data incrementally to ensure zero look-ahead bias.
3.  **Evaluation**: Assess the performance of the "trained" agent system on unseen data.


In [1]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

# Add project root to path
project_root = Path("../").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "FinAgents" / "orchestrator_demo"))
sys.path.insert(0, str(project_root / "FinAgents" / "agent_pools"))

from FinAgents.orchestrator_demo.orchestrator import Orchestrator

orchestrator = Orchestrator()
print("✅ Orchestrator Initialized")

2026-02-01 02:00:27,725 - ExecutionAgent - WARNING - Alpaca SDK not found. Using mock implementation.
2026-02-01 02:00:27,728 - Orchestrator - WARNING - alpaca-py not installed. Data fetching will be mocked.
2026-02-01 02:00:27,729 - ExecutionAgent - INFO - Using Mock Alpaca Service (invalid or mock keys detected)


⚠️  BacktestVisualizer not available
⚠️  Qlib not available - using simplified backtesting: No module named 'qlib'
✅ Orchestrator Initialized


### Step 1: Load Optimized Prompts

In [2]:
prompts_path = "optimized_prompts.json"

if os.path.exists(prompts_path):
    with open(prompts_path, "r") as f:
        optimized_prompts = json.load(f)
    
    # Apply prompts to agents
    orchestrator.alpha_agent.agent.instructions = optimized_prompts.get("Alpha", "")
    orchestrator.risk_agent.agent.instructions = optimized_prompts.get("Risk", "")
    orchestrator.portfolio_agent.agent.instructions = optimized_prompts.get("Portfolio", "")
    
    print("✅ Optimized prompts loaded and applied.")
else:
    print("⚠️ Optimized prompts file not found. Using default instructions.")

✅ Optimized prompts loaded and applied.


### Step 2: Run Out-of-Sample Test (World Model Mode)

We run the backtest for the future period (2024-2025) using **World Model Inference**. This iterates day-by-day, incrementally revealing data to the agent to ensure **zero look-ahead bias**.

In [3]:
symbol = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT']
# start_date = "2024-09-01"
start_date = "2024-12-01"
end_date = "2025-01-01" # Or today's date

print(f"🚀 Running Out-of-Sample Test for {symbol} ({start_date} to {end_date})...")

# Run Event-Driven World Model Inference (Step-by-Step)
print("🌍 Starting World Model Inference (Day-by-Day Simulation)...")
result = orchestrator.run_inference_rolling_week(symbol, start_date, end_date)

if result and result.get('status') == 'success':
    metrics = result.get('performance_metrics', {})
    print("\n📊 Out-of-Sample Performance Results:")
    print(f"   Total Return: {metrics.get('total_return', 0):.2%}")
    print(f"   Sharpe Ratio: {metrics.get('sharpe_ratio', 0):.2f}")
    print(f"   Max Drawdown: {metrics.get('max_drawdown', 0):.2%}")
else:
    print("❌ Test failed.")

2026-02-01 02:00:31,872 - Orchestrator - INFO - 🚀 Starting World Model Inference (Rolling Week) for ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT'] (2024-12-01 to 2025-01-01)
2026-02-01 02:00:31,888 - Orchestrator - INFO - Fetching data for ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT'] from 2023-12-02 00:00:00 to 2025-01-01 00:00:00
2026-02-01 02:00:31,895 - Orchestrator - WARNING - yfinance not installed. Falling back to mock data.
2026-02-01 02:00:31,913 - Orchestrator - INFO - 📅 Rolling Step: 2024-12-01 -> 2024-12-08 (Train: 2600, Test: 50)
2026-02-01 02:00:31,916 - Orchestrator - INFO - 📅 Rolling Step: 2024-12-08 -> 2024-12-15 (Train: 2600, Test: 50)
2026-02-01 02:00:31,918 - Orchestrator - INFO - 📅 Rolling Step: 2024-12-15 -> 2024-12-22 (Train: 2600, Test: 50)
2026-02-01 02:00:31,920 - Orchestrator - INFO - 📅 Rolling Step: 2024-12-22 -> 2024-12-29 (Train: 2600, Test: 50)
2026-02-01 02:00:31,929 - Orchestrator - INFO - 📅 R

🚀 Running Out-of-Sample Test for ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT'] (2024-12-01 to 2025-01-01)...
🌍 Starting World Model Inference (Day-by-Day Simulation)...
DEBUG: 🤖 Requesting Alpha Agent LLM...
DEBUG: LLM finished. Context keys: ['data', 'train_data', 'factors', 'indicators', 'model_type', 'signal_threshold', 'data_processor']
DEBUG: LLM response: <coroutine object Runner.run at 0x1117f89e0>
DEBUG: 🤖 Requesting Alpha Agent LLM...
DEBUG: LLM finished. Context keys: ['data', 'train_data', 'factors', 'indicators', 'model_type', 'signal_threshold', 'data_processor']
DEBUG: LLM response: <coroutine object Runner.run at 0x1117f89e0>
DEBUG: 🤖 Requesting Alpha Agent LLM...
DEBUG: LLM finished. Context keys: ['data', 'train_data', 'factors', 'indicators', 'model_type', 'signal_threshold', 'data_processor']
DEBUG: LLM response: <coroutine object Runner.run at 0x1117f89e0>
DEBUG: 🤖 Requesting Alpha Agent LLM...
DEBUG: LLM finished. Context keys: ['data

In [4]:
# Optional: Visualize if available
print("Result Details:", json.dumps(result, indent=2, default=str))

Result Details: {
  "status": "error",
  "message": "No signals generated during rolling inference"
}


In [ ]:
%tb